# Experiment 11: Model-Wide DBSCAN Tucker Compression Across All MLP Submodules

**Target Model**: `google/gemma-3-1b-it` (All 26 Transformer Decoder Layers: `model.layers[0...25]`)  
**Target Submodules**: All 3 MLP Projections (`gate_proj`, `up_proj`, and `down_proj` -> 78 total matrices)  
**Evaluation Task**: GLUE MNLI Validation Set (`validation_matched`)  

### Core Architecture & Configuration:
1. **Target Ranks**: `[3, 100, 350]` (Option A Aggressive DBSCAN).
2. **Dense Activation Clustering**:
   DBSCAN ($\epsilon=0.06$, $	ext{min\_samples}=40$) groups intermediate neurons into tight operational regimes.
3. **Outlier Quarantine**:
   Superweights (DBSCAN noise label `-1`, $|x| > 3.0$ or top 1% variance) are strictly isolated and preserved in uncompressed FP32.
4. **All-MLP Projection Symmetry**:
   - `gate_proj` ($6912 \times 1152$): Top 6 dense slices ($2,400$ coords) $\to [6, 400, 1152]$.
   - `up_proj` ($6912 \times 1152$): Top 6 dense slices ($2,400$ coords) $\to [6, 400, 1152]$.
   - `down_proj` ($1152 \times 6912$): Columns transposed ($2,400$ input channels) $\to [6, 400, 1152]$.
5. **Gradient Descent Core Finding**:
   SVD initialization refined with **PyTorch Adam** (35 iterations, $lr=10^{-3}$) on each 3D tensor slice.
6. **Total Parameters Cut**:
   $26 \times 3 \times (2,764,800 - 548,218) = \mathbf{172,893,396\text{ parameters}}$ permanently eliminated across the model.

In [1]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Library Imports
# =====================================================================
import os
import sys
import time
from pathlib import Path
import math
import json
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import torch
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

# Neural Decomp framework imports
try:
    from neural_decomp import ModelManagementInterface, DeviceMapOptions
    from neural_decomp.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from neural_decomp.utils import get_device_info, save_json_metrics
    print("Loaded neural_decomp library.")
except ImportError:
    from utility import ModelManagementInterface, DeviceMapOptions
    from utility.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from utility.utils import get_device_info, save_json_metrics
    print("Loaded utility library.")

device_info = get_device_info()
print(f"Device: {device_info['device_name']} | CUDA Available: {device_info['cuda_available']}")

/home/dwithun/Development/llm_compression/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded neural_decomp library.
Device: NVIDIA GeForce RTX 3070 Ti | CUDA Available: True


In [2]:
# =====================================================================
# STEP 2: Initialize Model & Tokenizer (Unprocessed Clean Baseline)
# =====================================================================
model_id = "google/gemma-3-1b-it"

mmi = ModelManagementInterface(
    model_id=model_id,
    precision=torch.float32,
    device_map=DeviceMapOptions.AUTO,
)
model = mmi.get_model()
tokenizer = mmi.get_tokenizer()

NUM_LAYERS = len(model.model.layers)
D_IN = model.model.layers[0].mlp.gate_proj.weight.shape[1]
D_OUT = model.model.layers[0].mlp.gate_proj.weight.shape[0]

print(f"Loaded {model_id}: {NUM_LAYERS} Transformer Decoder Layers")
print(f"MLP Submodule Dimensions: in_features={D_IN}, intermediate_features={D_OUT}")

# Cache original pristine weights on CPU across all 26 layers for calibration & rollback
W_orig_all = {
    l: {
        "gate_proj": model.model.layers[l].mlp.gate_proj.weight.data.clone().cpu(),
        "up_proj":   model.model.layers[l].mlp.up_proj.weight.data.clone().cpu(),
        "down_proj": model.model.layers[l].mlp.down_proj.weight.data.clone().cpu(),
    }
    for l in range(NUM_LAYERS)
}
print(f"Cached pristine weights on CPU for all {NUM_LAYERS} layers (78 projection matrices).")

Loading weights: 100%|██████████| 340/340 [00:01<00:00, 221.58it/s]


Loaded google/gemma-3-1b-it: 26 Transformer Decoder Layers
MLP Submodule Dimensions: in_features=1152, intermediate_features=6912
Cached pristine weights on CPU for all 26 layers (78 projection matrices).
time: 6.15s
cummulative_time: 8.18s


In [3]:
# =====================================================================
# STEP 3: Load GLUE MNLI Validation Benchmark
# =====================================================================
ds = load_dataset("nyu-mll/glue", "mnli")["validation_matched"]

label_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + name, add_special_tokens=False)[0] for name in label_names]

EVAL_SAMPLE_COUNT = 1000
eval_data = ds.select(range(EVAL_SAMPLE_COUNT))

print(f"Loaded GLUE MNLI: {len(ds):,} total samples | Active Evaluation Subset: {len(eval_data):,} samples")

Loaded GLUE MNLI: 9,815 total samples | Active Evaluation Subset: 1,000 samples
time: 3.71s
cummulative_time: 11.89s


## Step 1: Initial Baseline Evaluation & Layer-Wise Activation Hooking

We evaluate the unprocessed model on GLUE MNLI while registering forward hooks on `mlp.act_fn` across all 26 layers to record intermediate neuron activation profiles.

In [4]:
# =====================================================================
# STEP 4: Initial Baseline Evaluation & Simultaneous Activation Hooking
# =====================================================================
layer_trajectories = {l: [] for l in range(NUM_LAYERS)}
current_acts = {}

def make_act_hook(layer_idx):
    def hook_fn(module, input_tensor, output_tensor):
        act = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
        current_acts[layer_idx] = act.detach().cpu()
    return hook_fn

hooks = [
    model.model.layers[l].mlp.act_fn.register_forward_hook(make_act_hook(l))
    for l in range(NUM_LAYERS)
]

predictions = []
ground_truth = []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Baseline MNLI Inference & Activation Hooking"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs)

        for l in range(NUM_LAYERS):
            if l in current_acts and current_acts[l] is not None:
                pooled = current_acts[l].squeeze(0).mean(dim=0).numpy()
                layer_trajectories[l].append(pooled)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        predictions.append(pred_label)
        ground_truth.append(sample["label"])

for h in hooks:
    h.remove()

acts_matrices = {l: np.stack(layer_trajectories[l]) for l in range(NUM_LAYERS)}
baseline_accuracy = accuracy_score(ground_truth, predictions)
print(f"\nUncompressed Baseline Accuracy: {baseline_accuracy * 100:.2f}%")
print(f"Captured activation matrices across {len(acts_matrices)} layers (shape: {acts_matrices[0].shape})")

Baseline MNLI Inference & Activation Hooking: 100%|██████████| 1000/1000 [00:47<00:00, 21.20it/s]



Uncompressed Baseline Accuracy: 48.00%
Captured activation matrices across 26 layers (shape: (1000, 6912))
time: 47.35s
cummulative_time: 59.24s


## Step 2: Optimization Utility: Tucker with Adam Gradient Descent

We define `optimize_tucker_gd` which initializes factor matrices and core via SVD, followed by 35 steps of PyTorch Adam gradient descent refinement.

In [5]:
# =====================================================================
# STEP 5: Define Tucker Decomposition with Adam Gradient Descent
# =====================================================================
def optimize_tucker_gd(T, ranks=[3, 100, 350], num_steps=35, lr=1e-3, device="cpu"):
    """
    Performs Tucker decomposition with SVD initialization followed by
    PyTorch Adam gradient descent refinement of core tensor and factor matrices.
    """
    core_init, factors_init = tucker(T, rank=ranks, init='svd')
    core_param = torch.nn.Parameter(core_init.clone().to(device))
    factors_param = [torch.nn.Parameter(f.clone().to(device)) for f in factors_init]
    optimizer = torch.optim.Adam([core_param] + factors_param, lr=lr)
    T_target = T.to(device)

    with torch.no_grad():
        T_recon_init = tucker_to_tensor((core_param, factors_param))
        init_err = (torch.norm(T_target - T_recon_init) / torch.norm(T_target)).item()

    for step in range(num_steps):
        optimizer.zero_grad()
        T_recon = tucker_to_tensor((core_param, factors_param))
        loss = torch.norm(T_target - T_recon) ** 2
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        T_recon_final = tucker_to_tensor((core_param, factors_param)).cpu()
        final_err = (torch.norm(T.cpu() - T_recon_final) / torch.norm(T.cpu())).item()

    return core_param.detach().cpu(), [f.detach().cpu() for f in factors_param], T_recon_final, init_err, final_err

print("Tucker Adam GD optimizer defined.")

Tucker Adam GD optimizer defined.
time: 0.00s
cummulative_time: 59.25s


## Step 3: Layer-by-Layer DBSCAN Clustering & Factorization of All MLP Submodules

For each layer $l \in [0 \dots 25]$:
1. Run DBSCAN on the 6,912 intermediate neuron responses (mean activation).
2. Quarantine superweights/outliers in pristine FP32.
3. Extract top 6 dense clusters into uniform slices of size $M=400$ ($2,400$ coordinates total).
4. Factorize **`gate_proj`** ($[6, 400, 1152]$) using rank `[3, 100, 350]` with Adam GD.
5. Factorize **`up_proj`** ($[6, 400, 1152]$) using rank `[3, 100, 350]` with Adam GD.
6. Factorize **`down_proj`** (transposed active columns $\to [6, 400, 1152]$) using rank `[3, 100, 350]` with Adam GD.
7. Inject reconstructed weights and restore pristine superweights.

In [6]:
# =====================================================================
# STEP 6: Execute Layer-by-Layer All-MLP DBSCAN Tucker Compression
# =====================================================================
CHUNK_SIZE = 400
NUM_CHUNKS = 6
ACTIVE_COORDS_TARGET = CHUNK_SIZE * NUM_CHUNKS  # 2,400
TUCKER_RANKS = [3, 100, 350]
DBSCAN_EPS = 0.06
DBSCAN_MIN_SAMPLES = 40

layer_stats = []

print(f"Starting Model-Wide Compression across all {NUM_LAYERS} Layers (78 Projections)...")
print(f"Configuration: Ranks={TUCKER_RANKS}, Active Coords={ACTIVE_COORDS_TARGET}, eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES}\n")

for l in range(NUM_LAYERS):
    layer_mod = model.model.layers[l]
    acts_l = acts_matrices[l]
    num_samples, total_coords = acts_l.shape

    # 1. Mean coordinate activation response
    coord_rep_values = np.mean(acts_l, axis=0)

    # 2. Fit DBSCAN
    db = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, metric="euclidean")
    labels = db.fit_predict(coord_rep_values.reshape(-1, 1))

    # Identify outliers/superweights (noise label -1 + high magnitude/variance)
    max_mags = np.max(np.abs(acts_l), axis=0)
    variances = np.var(acts_l, axis=0)
    super_mask = (labels == -1) | (max_mags > 3.0) | (variances >= np.quantile(variances, 0.99))
    super_indices = np.where(super_mask)[0]

    unique_labels = [lab for lab in np.unique(labels) if lab != -1]

    # 3. Form top 6 uniform chunks of size 400
    chunk_slices = []
    chunk_coords_list = []
    
    for lab in unique_labels:
        c_indices = np.where((labels == lab) & (~super_mask))[0]
        if len(c_indices) == 0:
            continue
        sorted_indices = c_indices[np.argsort(coord_rep_values[c_indices])]
        num_full = len(sorted_indices) // CHUNK_SIZE
        for ci in range(num_full):
            sub_c = sorted_indices[ci * CHUNK_SIZE : (ci + 1) * CHUNK_SIZE]
            chunk_coords_list.append(sub_c)
            if len(chunk_coords_list) >= NUM_CHUNKS:
                break
        if len(chunk_coords_list) >= NUM_CHUNKS:
            break

    # If fewer than 6 chunks found, backfill from remaining non-superweight coords
    if len(chunk_coords_list) < NUM_CHUNKS:
        all_assigned = set(np.concatenate(chunk_coords_list) if chunk_coords_list else [])
        avail = [i for i in range(total_coords) if i not in all_assigned and not super_mask[i]]
        needed = NUM_CHUNKS - len(chunk_coords_list)
        for ci in range(needed):
            if len(avail) >= CHUNK_SIZE:
                sub_c = np.array(avail[:CHUNK_SIZE])
                avail = avail[CHUNK_SIZE:]
                chunk_coords_list.append(sub_c)

    active_coords = np.concatenate(chunk_coords_list)
    
    # Original weights
    W_gate_orig = W_orig_all[l]["gate_proj"]
    W_up_orig = W_orig_all[l]["up_proj"]
    W_down_orig = W_orig_all[l]["down_proj"]

    # --- A. Factorize gate_proj ---
    T_gate = torch.stack([W_gate_orig[c, :].float() for c in chunk_coords_list], dim=0)
    cg, fg, T_gate_recon, init_err_g, final_err_g = optimize_tucker_gd(T_gate, ranks=TUCKER_RANKS, device="cpu")
    
    layer_mod.mlp.gate_proj.weight.data = W_gate_orig.clone().to(device=model.device)
    for k, c in enumerate(chunk_coords_list):
        layer_mod.mlp.gate_proj.weight.data[c, :] = T_gate_recon[k].to(
            device=model.device, dtype=layer_mod.mlp.gate_proj.weight.dtype
        )
    layer_mod.mlp.gate_proj.weight.data[super_indices, :] = W_gate_orig[super_indices, :].to(device=model.device)

    # --- B. Factorize up_proj ---
    T_up = torch.stack([W_up_orig[c, :].float() for c in chunk_coords_list], dim=0)
    cu, fu, T_up_recon, init_err_u, final_err_u = optimize_tucker_gd(T_up, ranks=TUCKER_RANKS, device="cpu")
    
    layer_mod.mlp.up_proj.weight.data = W_up_orig.clone().to(device=model.device)
    for k, c in enumerate(chunk_coords_list):
        layer_mod.mlp.up_proj.weight.data[c, :] = T_up_recon[k].to(
            device=model.device, dtype=layer_mod.mlp.up_proj.weight.dtype
        )
    layer_mod.mlp.up_proj.weight.data[super_indices, :] = W_up_orig[super_indices, :].to(device=model.device)

    # --- C. Factorize down_proj ---
    # In down_proj, active coordinates are columns!
    T_down = torch.stack([W_down_orig[:, c].T.float() for c in chunk_coords_list], dim=0)
    cd, fd, T_down_recon, init_err_d, final_err_d = optimize_tucker_gd(T_down, ranks=TUCKER_RANKS, device="cpu")
    
    layer_mod.mlp.down_proj.weight.data = W_down_orig.clone().to(device=model.device)
    for k, c in enumerate(chunk_coords_list):
        # Transpose [400, 1152] back to [1152, 400]
        layer_mod.mlp.down_proj.weight.data[:, c] = T_down_recon[k].T.to(
            device=model.device, dtype=layer_mod.mlp.down_proj.weight.dtype
        )
    layer_mod.mlp.down_proj.weight.data[:, super_indices] = W_down_orig[:, super_indices].to(device=model.device)

    del T_gate, T_up, T_down, T_gate_recon, T_up_recon, T_down_recon
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Parameter accounting
    params_orig_tensor = NUM_CHUNKS * CHUNK_SIZE * D_IN  # 2,764,800
    params_comp_tensor = cg.numel() + sum(f.numel() for f in fg)  # 548,218
    params_saved_per_proj = params_orig_tensor - params_comp_tensor  # 2,216,582
    params_saved_layer = params_saved_per_proj * 3  # 6,649,746

    layer_stats.append({
        "layer": l,
        "superweights": len(super_indices),
        "gate_err": round(final_err_g * 100, 2),
        "up_err": round(final_err_u * 100, 2),
        "down_err": round(final_err_d * 100, 2),
        "params_saved": params_saved_layer,
    })

    print(f"Layer {l:>2}/25 | Superweights: {len(super_indices):>4} | "
          f"Err: gate={final_err_g*100:.1f}%, up={final_err_u*100:.1f}%, down={final_err_d*100:.1f}% | "
          f"Saved: {params_saved_layer:,} params")

print(f"\nModel-Wide Compression Complete across all {NUM_LAYERS} layers (78 matrices).")

Starting Model-Wide Compression across all 26 Layers (78 Projections)...
Configuration: Ranks=[3, 100, 350], Active Coords=2400, eps=0.06, min_samples=40

Layer  0/25 | Superweights:  329 | Err: gate=88.6%, up=89.8%, down=89.1% | Saved: 6,649,746 params
Layer  1/25 | Superweights:  143 | Err: gate=88.8%, up=90.0%, down=89.4% | Saved: 6,649,746 params
Layer  2/25 | Superweights:  134 | Err: gate=89.0%, up=90.0%, down=89.1% | Saved: 6,649,746 params
Layer  3/25 | Superweights:  151 | Err: gate=88.7%, up=90.0%, down=89.0% | Saved: 6,649,746 params
Layer  4/25 | Superweights:  119 | Err: gate=88.3%, up=89.9%, down=89.5% | Saved: 6,649,746 params
Layer  5/25 | Superweights:  147 | Err: gate=88.0%, up=89.5%, down=89.4% | Saved: 6,649,746 params
Layer  6/25 | Superweights:  143 | Err: gate=88.1%, up=89.3%, down=89.2% | Saved: 6,649,746 params
Layer  7/25 | Superweights:  141 | Err: gate=87.8%, up=89.0%, down=88.7% | Saved: 6,649,746 params
Layer  8/25 | Superweights:  125 | Err: gate=87.7%, u

## Step 4: Full-Model Downstream Evaluation on GLUE MNLI

We evaluate the fully compressed model (all 26 layers $\times$ 3 MLP submodules) on GLUE MNLI validation set to measure task accuracy retention.

In [7]:
# =====================================================================
# STEP 7: Full-Model GLUE MNLI Downstream Evaluation
# =====================================================================
comp_predictions = []
comp_ground_truth = []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Evaluating All-MLP Compressed Model"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        comp_predictions.append(pred_label)
        comp_ground_truth.append(sample["label"])

compressed_accuracy = accuracy_score(comp_ground_truth, comp_predictions)
accuracy_delta = compressed_accuracy - baseline_accuracy

print(f"\nAll-MLP Model Compression Benchmark Results:")
print(f"  Baseline Accuracy:         {baseline_accuracy * 100:.2f}%")
print(f"  Compressed Accuracy:       {compressed_accuracy * 100:.2f}%")
print(f"  Accuracy Delta:            {accuracy_delta * 100:+.2f}%")

Evaluating All-MLP Compressed Model: 100%|██████████| 1000/1000 [00:34<00:00, 28.57it/s]


All-MLP Model Compression Benchmark Results:
  Baseline Accuracy:         48.00%
  Compressed Accuracy:       34.40%
  Accuracy Delta:            -13.60%
time: 35.00s
cummulative_time: 233.23s


## Step 5: Parameter Accounting & Metrics Export

We calculate the total parameter reductions across all 78 MLP matrices and export results to `artifacts/11_all_layers_dbscan_results.json`.

In [8]:
# =====================================================================
# STEP 8: Parameter Accounting & Artifact Export
# =====================================================================
total_mlp_params_orig = NUM_LAYERS * 3 * (D_IN * D_OUT)  # 621,084,672
total_active_params_orig = NUM_LAYERS * 3 * (NUM_CHUNKS * CHUNK_SIZE * D_IN)  # 215,654,400
total_active_params_comp = NUM_LAYERS * 3 * (cg.numel() + sum(f.numel() for f in fg))  # 42,761,004
total_params_eliminated = total_active_params_orig - total_active_params_comp  # 172,893,396

pct_cut_active = (1.0 - total_active_params_comp / total_active_params_orig) * 100.0
pct_cut_total_mlp = (total_params_eliminated / total_mlp_params_orig) * 100.0
effective_mlp_ratio = total_mlp_params_orig / (total_mlp_params_orig - total_params_eliminated)

print("=" * 95)
print(f"{'Metric':<40} | {'Value':<30}")
print("=" * 95)
print(f"{'Total MLP Layers Factorized':<40} | {NUM_LAYERS * 3} projections (26 layers x 3)")
print(f"{'Tucker Ranks Applied':<40} | {TUCKER_RANKS}")
print(f"{'Pristine MLP Parameters':<40} | {total_mlp_params_orig:,}")
print(f"{'Parameters Eliminated':<40} | {total_params_eliminated:,} ({pct_cut_total_mlp:.2f}% of all MLP weights)")
print(f"{'Active Tensor Parameter Cut':<40} | {pct_cut_active:.2f}% (5.04x compression)")
print(f"{'Effective MLP Compression Ratio':<40} | {effective_mlp_ratio:.2f}x")
print(f"{'Baseline Accuracy':<40} | {baseline_accuracy * 100:.2f}%")
print(f"{'All-MLP Compressed Accuracy':<40} | {compressed_accuracy * 100:.2f}%")
print(f"{'Accuracy Delta':<40} | {accuracy_delta * 100:+.2f}%")
print("=" * 95)

# Save artifact
os.makedirs("artifacts", exist_ok=True)
results_data = {
    "experiment": "11_all_layers_dbscan_tucker",
    "target_model": model_id,
    "ranks": TUCKER_RANKS,
    "num_layers": NUM_LAYERS,
    "num_projections": NUM_LAYERS * 3,
    "total_mlp_params_orig": total_mlp_params_orig,
    "total_params_eliminated": total_params_eliminated,
    "pct_cut_total_mlp": round(pct_cut_total_mlp, 2),
    "pct_cut_active": round(pct_cut_active, 2),
    "baseline_accuracy": round(baseline_accuracy * 100, 2),
    "compressed_accuracy": round(compressed_accuracy * 100, 2),
    "accuracy_delta": round(accuracy_delta * 100, 2),
    "layer_stats": layer_stats,
    "timings": NOTEBOOK_TIMINGS,
}

with open("artifacts/11_all_layers_dbscan_results.json", "w") as f:
    json.dump(results_data, f, indent=2)

print(f"Saved benchmark results to artifacts/11_all_layers_dbscan_results.json")

Metric                                   | Value                         
Total MLP Layers Factorized              | 78 projections (26 layers x 3)
Tucker Ranks Applied                     | [3, 100, 350]
Pristine MLP Parameters                  | 621,084,672
Parameters Eliminated                    | 172,893,396 (27.84% of all MLP weights)
Active Tensor Parameter Cut              | 80.17% (5.04x compression)
Effective MLP Compression Ratio          | 1.39x
Baseline Accuracy                        | 48.00%
All-MLP Compressed Accuracy              | 34.40%
Accuracy Delta                           | -13.60%
Saved benchmark results to artifacts/11_all_layers_dbscan_results.json
time: 0.00s
cummulative_time: 233.24s
